In [1]:
import os, sys
sys.path.append(os.path.abspath('../..'))
from utlis.sync_utlis.general_loader import load_flat_with_frame_map, merge_pred_with_miniscope
from utlis.sync_utlis.general_loader_viz import plot_two_coms_from_pred_df


oct3v1 = "/data/big_rim/rsync_dcc_sum/Oct3V1"
rec_path = os.path.join(oct3v1, "2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30")
nc_key= "wnd1000_stp700_max25_diff3.5_pnrauto"
# 'wnd1500_stp700_max25_diff3.5_pnr1.1'
dannce_folder='SDANNCE/predict00'


merged = merge_pred_with_miniscope(
    rec_path=rec_path,
    nc_key=nc_key,
    dannce_folder=dannce_folder,
    com_folder=None,
    save_h5=False,
    save_csv=False,
)

merged.shape, merged.index[:3]

((12700, 237), Index([-5, 35, 64], dtype='int64', name='timestamp_ms_mini'))

In [2]:
from utlis.social_analysis.approaching import compute_com_distance, compute_motion_direction, detect_approaches
# 0) If you want distance only
dist = compute_com_distance(merged, p1="com1", p2="com2", smooth_window=5)
print(dist.head())

# 1) Motion vectors / directions for each animal
m1 = compute_motion_direction(merged, prefix="com1", time_col="timestamp_ms_mini", pos_smooth=5, vel_smooth=5)
m2 = compute_motion_direction(merged, prefix="com2", time_col="timestamp_ms_mini", pos_smooth=5, vel_smooth=5)
print(m1.filter(like="_speed").head(), m2.filter(like="_speed").head())

# 2) Detect approach windows (tune thresholds as you like)
res = detect_approaches(
    merged,
    p1="com1", p2="com2",
    time_col="timestamp_ms_mini",   # if absent, pass fps=...
    pos_smooth=5, vel_smooth=5,
    radial_thresh=20.0,             # mm/s toward the other
    speed_min=5.0,                  # mm/s total speed floor
    dist_min=None,                  # e.g., set 50.0 to ignore near-collisions if needed
    dist_max=300.0,                 # focus on interaction zone
    min_samples=15,                 # ≥15 consecutive frames
    return_intervals=True
)

# Per-frame booleans + metrics
res["frames"][["dist_mm","radial1","radial2","approach1","approach2","mutual"]].head()

# Interval summaries (list of dicts)
res["intervals"]["approach1"][:3], res["intervals"]["approach2"][:3], res["intervals"]["mutual"][:3]


timestamp_ms_mini
-5      50.580713
 35     50.694257
 64     50.803232
 97     50.946379
 130    51.173287
Name: dist_mm, dtype: float64
                   com1_speed
timestamp_ms_mini            
-5                   0.154367
 35                  0.162126
 64                  0.167439
 97                  0.159762
 130                 0.147180                    com2_speed
timestamp_ms_mini            
-5                   0.047668
 35                  0.057717
 64                  0.065080
 97                  0.077614
 130                 0.081325


([], [], [])

In [3]:
def pick_alignment_cols(frames):
    frame_col = None
    for c in ("camera_frame_sixcam", "mapped_sixcam_frame_indices"):
        if c in frames.columns:
            frame_col = c
            break
    mini_ms_col = "timestamp_ms_mini" if "timestamp_ms_mini" in frames.columns else None
    return frame_col, mini_ms_col


def extract_incidents(frames, events, frame_col=None, mini_ms_col=None):
    """
    Turns events into simple per-event lists you can feed to other funcs.
    Returns a list of dicts:
      {"start_idx", "end_idx_exclusive", "index", "sixcam", "mini_ms"}
    Missing cols are omitted.
    """
    out = []
    for ev in events:
        s, e = ev["start_idx"], ev["end_idx_exclusive"]
        sl = slice(s, e)
        item = {
            "start_idx": s,
            "end_idx_exclusive": e,
            "index": frames.index[sl].tolist()
        }
        if frame_col:
            item["sixcam"] = frames.iloc[sl][frame_col].astype(int).tolist()
        if mini_ms_col:
            item["mini_ms"] = frames.iloc[sl][mini_ms_col].astype(float).tolist()
        out.append(item)
    return out


import numpy as np
import matplotlib.pyplot as plt

def plot_dist_with_events(frames, mask, events, contact_mm=50.0, title=None):
    dist = frames["dist_mm"].to_numpy()
    x = frames.index

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(x, dist, lw=1, label="dist_mm")
    ax.scatter(x[mask], dist[mask], s=8, label="approach_success", zorder=3)
    ax.axhline(contact_mm, ls="--", lw=1, label=f"contact={contact_mm} mm")

    if events:
        contact_idx = np.array([ev["contact_idx"] for ev in events], dtype=int)
        ax.scatter(x[contact_idx], dist[contact_idx], marker="x", s=36, zorder=4, label="first contact")

        # light spans per event (fast; number of events is small)
        for ev in events:
            s, e = ev["start_idx"], ev["end_idx_exclusive"]
            ax.axvspan(x[s], x[e-1], alpha=0.15)

    ax.set_ylabel("distance (mm)")
    ax.set_xlabel("frame")  # or "time" if your index is time-like
    ax.set_title(title or f"Approach-success events (n={len(events)})")
    ax.legend(loc="best")
    plt.tight_layout()
    return fig, ax


# def plot_single_event(frames, ev, contact_mm=50.0):
#     s, e = ev["start_idx"], ev["end_idx_exclusive"]
#     sl = slice(s, e)
#     x = frames.index[sl]
#     dist = frames["dist_mm"].iloc[sl]

#     fig, ax = plt.subplots(figsize=(8, 3))
#     ax.plot(x, dist, lw=1.2)
#     ax.axhline(contact_mm, ls="--", lw=1)
#     # mark first contact if ev has it
#     if "contact_idx" in ev:
#         cx = frames.index[ev["contact_idx"]]
#         ax.scatter([cx], [frames["dist_mm"].iloc[ev["contact_idx"]]], marker="x", s=36, zorder=4)
#     ax.set_ylabel("distance (mm)")
#     ax.set_xlabel("frame")
#     ax.set_title(f"Event [{s}:{e})  drop={ev.get('drop_mm', np.nan):.1f}mm")
#     plt.tight_layout()
#     return fig, ax

def plot_single_event(frames, ev, contact_mm=50.0, mark="both"):
    """
    mark: "contact" | "bottom" | "both"
    """
    s, e = ev["start_idx"], ev["end_idx_exclusive"]
    x = frames.index[s:e]
    dist = frames["dist_mm"].iloc[s:e]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(x, dist, lw=1.2, zorder=1)
    ax.axhline(contact_mm, ls="--", lw=1, zorder=0)

    # start/end verticals for context
    ax.axvline(frames.index[s], ls=":", lw=1, alpha=0.6, zorder=0)
    ax.axvline(frames.index[e-1], ls=":", lw=1, alpha=0.6, zorder=0)

    if ("contact_idx" in ev) and (mark in {"contact", "both"}):
        ci = ev["contact_idx"]
        ax.scatter([frames.index[ci]], [frames["dist_mm"].iat[ci]],
                   marker="x", s=40, zorder=3, label="first contact")

    if ("bottom_idx" in ev) and (mark in {"bottom", "both"}):
        bi = ev["bottom_idx"]
        ax.scatter([frames.index[bi]], [frames["dist_mm"].iat[bi]],
                   marker="o", s=36, zorder=4, label="bottom (min distance)")

    ax.set_ylabel("distance (mm)")
    ax.set_xlabel("frame")
    ax.set_title(f"Event [{s}:{e})  drop={ev.get('drop_mm', float('nan')):.1f} mm")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    return fig, ax



In [4]:
from utlis.social_analysis.approaching import _boolean_runs, find_approach_success


contact_mm = 50
# --- choose a setting (use your "best" from the sweep) ---
mask, events = find_approach_success(
    res["frames"],
    contact_mm=contact_mm,
    dD_dt_thresh=0.0,
    min_len=1,
    min_drop_mm=1#10.0
)

# mask0, events0 = find_approach_success(res["frames"], contact_mm=50, dD_dt_thresh=0.0, min_len=1, min_drop_mm=1)
# mask, events = merge_overlapping_approach_events(res["frames"], events0, contact_mm=150, max_gap=0, min_len=1, min_drop_mm=1)

frames = res["frames"]

# --- global plot with event spans (uses the helper we wrote) ---
# plot_dist_with_events(frames, mask, events, contact_mm=contact_mm)

# --- quick QA: first 3 events as small windows ---
# for ev in events[:3]:
#     plot_single_event(frames, ev, contact_mm=contact_mm)

In [5]:
# from utlis.social_analysis.approaching import find_approach_success
from utlis.vis_valid_utlis.sdannce_vis import cfg, visualize_frames

C = cfg(base_path="/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30", cammm=1, 
        enable_zoom=True, zoom_margin=450, video_fps=30) #animal="mouse20", #write_pngs=True

# 0) Run your detector first
mask, events = find_approach_success(
    res["frames"],
    contact_mm=contact_mm,
    dD_dt_thresh=0.0,
    min_len=1,
    min_drop_mm=1.0
)

frames = res["frames"]
frame_col = "camera_frame_sixcam" if "camera_frame_sixcam" in frames.columns \
    else ("mapped_sixcam_frame_indices" if "mapped_sixcam_frame_indices" in frames.columns else None)

print(f"#approach_success events: {len(events)}")

incident_all_six_cam = frames.loc[mask, frame_col].astype(int).tolist() if frame_col else frames.index[mask].tolist()

# visualize_frames(incident_all_six_cam, config=C, out_name=f"251005_specifc_frame_all_incidents_contact{contact_mm}_dD0_min1d1")

#approach_success events: 4


In [ ]:
# from typing import List, Optional, Any
# import os
# import numpy as np
# import imageio
# import matplotlib.pyplot as plt
# from matplotlib.animation import FFMpegWriter
# from matplotlib.gridspec import GridSpec
# import tqdm

# # --- your existing modules ---
# from utlis import connectivity

# # Prefer pulling helpers from the same place as your original visualize_frames:
# try:
#     from utlis.vis_valid_utlis.sdannce_vis import (
#         _apply_overrides, _prepare_io,
#         _load_and_project_preds_frames, _load_and_project_com_frames,
#         _draw_frame, DEFAULT_CFG, load_cameras,
#     )
# except ImportError as e:
#     raise ImportError(
#         "Couldn't import visualize_frames helpers from utlis.vis_valid_utlis.sdannce_vis. "
#         "Point these imports to wherever your originals live."
#     ) from e

# # Optional: 3D plotter and 3D DF loader — try a few likely spots, but both are also
# # accepted as function args (df3d / plotter_3d) so you can bypass imports entirely.
# try:
#     from utlis.plotting import plot_skeleton_frame_3d_const_colors as _DEFAULT_PLOTTER_3D
# except Exception:
#     _DEFAULT_PLOTTER_3D = None

# def _load_3d_dataframe_default(label3d_path):
#     """Override by passing df3d=... to the function, or replace this import with your own."""
#     try:
#         # If you already have a loader alongside your other helpers, import it there:
#         from utlis.vis_valid_utlis.sdannce_vis import load_3d_dataframe
#         return load_3d_dataframe(label3d_path)
#     except Exception as e:
#         raise ImportError(
#             "No 3D dataframe loader found. Pass df3d=... to visualize_frames_triptych "
#             "or expose `load_3d_dataframe(label3d_path)` in your project."
#         ) from e


# def visualize_frames_triptych(
#     frame_list: List[int],
#     config: Optional['VizConfig'] = None,
#     *,
#     df3d=None,
#     plotter_3d=None,
#     **overrides: Any
# ) -> None:
#     """
#     Three-panel renderer (Top: raw | Middle: 3D | Bottom: 2D overlay).
#     Usage mirrors your existing visualize_frames(...).

#     You can pass df3d=... (preloaded 3D DataFrame) and/or plotter_3d=...
#     to avoid hunting for import paths.
#     """
#     if not frame_list:
#         raise ValueError("frame_list is empty.")
#     frames = list(frame_list)

#     c = _apply_overrides(config or DEFAULT_CFG, overrides)

#     cam, video_path, label3d_path, pred_path, com_path, save_path = _prepare_io(c)
#     COLOR = connectivity.COLOR_DICT[c.animal]
#     CONNECTIVITY = connectivity.CONNECTIVITY_DICT[c.animal]
#     cameras = load_cameras(label3d_path)

#     # 2D sparse loads for overlay
#     pred_2d = _load_and_project_preds_frames(pred_path, cameras, cam, frames)
#     pred_2d_com = _load_and_project_com_frames(com_path, cameras, cam, frames)

#     # 3D source
#     if df3d is None:
#         df3d = _load_3d_dataframe_default(label3d_path)

#     # 3D plotter
#     plotter_3d = plotter_3d or _DEFAULT_PLOTTER_3D
#     if plotter_3d is None:
#         raise ImportError(
#             "3D plotter not found. Provide plotter_3d=plot_skeleton_frame_3d_const_colors "
#             "or expose it from utlis.plotting."
#         )

#     # map absolute video frame -> df3d index
#     def _map_frame_to_df_idx(f_abs):
#         if hasattr(c, "frame_to_df_index") and callable(c.frame_to_df_index):
#             try:
#                 return c.frame_to_df_index(f_abs)
#             except Exception:
#                 pass
#         try:
#             if f_abs in df3d.index:
#                 return f_abs
#         except Exception:
#             pass
#         if "frame_idx" in getattr(df3d, "columns", []):
#             m = np.flatnonzero(df3d["frame_idx"].to_numpy() == int(f_abs))
#             if m.size:
#                 return df3d.index[m[0]]
#         if getattr(c, "allow_nearest_idx", False):
#             try:
#                 idx_vals = np.asarray(df3d.index, dtype=float)
#                 j = int(np.nanargmin(np.abs(idx_vals - float(f_abs))))
#                 return df3d.index[j]
#             except Exception:
#                 return None
#         return None

#     # figure/axes
#     fig = plt.figure(figsize=(6.5, 9.5))
#     gs = GridSpec(3, 1, figure=fig, height_ratios=[1.0, 1.05, 1.0], hspace=0.08)
#     ax_top = fig.add_subplot(gs[0, 0])                   # raw
#     ax_mid = fig.add_subplot(gs[1, 0], projection='3d')  # 3D
#     ax_bot = fig.add_subplot(gs[2, 0])                   # overlay

#     # optional panel labels
#     if getattr(c, "panel_labels", None):
#         ax_top.set_title(str(c.panel_labels[0]), fontsize=10)
#         ax_mid.set_title(str(c.panel_labels[1]), fontsize=10)
#         ax_bot.set_title(str(c.panel_labels[2]), fontsize=10)

#     vids = imageio.get_reader(video_path)

#     tag = f"custom_frames_{frames[0]}_{frames[-1]}_{len(frames)}f"
#     vid_name = (c.out_name if c.out_name else f"triptych_cam{c.cammm}_{tag}") + ".mp4"
#     out_mp4 = os.path.join(save_path, "vis_" + vid_name)
#     writer = FFMpegWriter(fps=c.video_fps, metadata=dict(title='triptych_visualization', artist='Matplotlib'))

#     fixed_3d = getattr(c, "fixed_limits_3d", None)
#     animal1 = getattr(c, "animal1", "a1")
#     animal2 = getattr(c, "animal2", "a2")

#     with writer.saving(fig, out_mp4, dpi=getattr(c, "dpi", 300)):
#         for j, f_abs in enumerate(tqdm.tqdm(frames)):
#             # -------- top: raw --------
#             img = vids.get_data(f_abs)
#             ax_top.cla()
#             ax_top.imshow(img)
#             ax_top.set_axis_off()
#             ax_top.text(0.01, 0.98, f"cam{c.cammm} • frame {f_abs}",
#                         transform=ax_top.transAxes, va="top", ha="left",
#                         fontsize=9, bbox=dict(boxstyle="round,pad=0.2", fc="0.0", ec="none", alpha=0.3), color="w")

#             # -------- middle: 3D --------
#             ax_mid.cla()
#             if fixed_3d:
#                 if "xlim" in fixed_3d: ax_mid.set_xlim(*fixed_3d["xlim"])
#                 if "ylim" in fixed_3d: ax_mid.set_ylim(*fixed_3d["ylim"])
#                 if "zlim" in fixed_3d: ax_mid.set_zlim(*fixed_3d["zlim"])
#             try:
#                 ax_mid.set_box_aspect((1, 1, 1))
#             except Exception:
#                 pass

#             df_idx = _map_frame_to_df_idx(f_abs)
#             plt.sca(ax_mid)
#             if df_idx is not None and (df_idx in df3d.index):
#                 plotter_3d(
#                     df3d, df_idx,
#                     COLOR=COLOR, CONNECTIVITY=CONNECTIVITY,
#                     animal1=animal1, animal2=animal2,
#                     elev=getattr(c, "elev", 22),
#                     azim=getattr(c, "azim", -55),
#                     invert_y=getattr(c, "invert_y", False),
#                     kp_size=getattr(c, "kp_size", 14),
#                     lw=getattr(c, "lw", 2.0),
#                     dpi=getattr(c, "dpi", 300),
#                 )
#             else:
#                 ax_mid.text(0.5, 0.5, "3D unavailable for this frame",
#                             transform=ax_mid.transAxes, ha="center", va="center", fontsize=10)
#                 ax_mid.set_axis_off()

#             # -------- bottom: 2D overlay --------
#             ax_bot.cla()
#             plt.sca(ax_bot)  # _draw_frame uses current axes
#             k = pred_2d[cam][j]
#             k_a1, k_a2 = (k, None) if k.ndim == 2 else (k[0], k[1])
#             k_com = pred_2d_com[cam][j]

#             _draw_frame(
#                 img, k_a1, k_a2, k_com,
#                 COLOR, CONNECTIVITY,
#                 enable_zoom=getattr(c, "enable_zoom", True),
#                 zoom_margin=getattr(c, "zoom_margin", 80),
#                 drop_tail_for_view=getattr(c, "drop_tail_for_view", False),
#                 include_both=getattr(c, "zoom_include_both", True),
#                 title_text=None
#             )

#             writer.grab_frame()

#             if getattr(c, "write_pngs", False):
#                 fig.savefig(
#                     os.path.join(save_path, f"vis3_triptych_frame_{f_abs}.png"),
#                     dpi=getattr(c, "dpi", 300), bbox_inches="tight"
#                 )
